In [4]:
import os
from tqdm import tqdm
def remove_none_class_and_reindex(labels_dir, old_num_classes):
    """
    labels_dir: папка, где лежат YOLO-аннотации (.txt)
    old_num_classes: исходное число классов (включая None)
    """
    # Пробежимся по всем .txt-файлам
    for file_name in tqdm(os.listdir(labels_dir)):
        if not file_name.endswith(".txt"):
            continue
        
        file_path = os.path.join(labels_dir, file_name)
        with open(file_path, "r") as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            # print(f"before line: {line}")
            parts = line.strip().split()
            cls_id = int(parts[0])
            # Если класс == 0 (None), пропускаем
            if cls_id == 0:
                continue
            # Иначе смещаем класс на -1, чтобы 1 -> 0, 2 -> 1 и т.д.
            new_cls_id = cls_id - 1
            # Собираем строку обратно
            new_line = f"{new_cls_id} " + " ".join(parts[1:]) + "\n"
            # print(f"after line: {new_line}")
            new_lines.append(new_line)
        
        # Перезаписываем файл без строк класса 0 и со смещёнными индексами
        with open(file_path, "w") as f:
            f.writelines(new_lines)

# Пример использования:
# Предположим, раньше у вас было 14 классов (0=none, 1..13=остальные),
# значит old_num_classes = 14.
labels_dir = "G:/researchwork/leykocytes_dataset/labels"
remove_none_class_and_reindex(labels_dir, old_num_classes=14)


100%|██████████| 7099/7099 [00:29<00:00, 238.16it/s]


In [6]:
import os
import random
import shutil
from tqdm import tqdm

def split_dataset(
    base_dir,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42
):
    """
    base_dir: путь к папке leykocytes_dataset, внутри которой лежат папки images и labels.
    train_ratio, val_ratio, test_ratio: доли разбиения.
    seed: зерно для воспроизводимости.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Сумма долей train, val, test должна быть равна 1."

    images_dir = os.path.join(base_dir, "images")
    labels_dir = os.path.join(base_dir, "labels")

    # Проверяем, что папки с изображениями и метками существуют
    if not os.path.isdir(images_dir) or not os.path.isdir(labels_dir):
        raise FileNotFoundError("Не найдены папки 'images' или 'labels' в {}".format(base_dir))

    # Список всех файлов изображений
    all_images = sorted(os.listdir(images_dir))

    # Фильтруем только файлы с расширениями (например .jpg, .png и т.д.)
    valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    all_images = [img for img in all_images if img.lower().endswith(valid_exts)]

    # Перемешиваем файлы для случайного разбиения
    random.seed(seed)
    random.shuffle(all_images)

    # Считаем, сколько пойдёт в train, val, test
    total = len(all_images)
    train_count = int(total * train_ratio)
    val_count = int(total * val_ratio)
    test_count = total - train_count - val_count  # остаток на test

    train_images = all_images[:train_count]
    val_images = all_images[train_count:train_count+val_count]
    test_images = all_images[train_count+val_count:]

    # Создадим папки для train, val, test
    for split in ["train", "val", "test"]:
        split_images_dir = os.path.join(base_dir, split, "images")
        split_labels_dir = os.path.join(base_dir, split, "labels")
        os.makedirs(split_images_dir, exist_ok=True)
        os.makedirs(split_labels_dir, exist_ok=True)

    # Вспомогательная функция для копирования
    def copy_files(img_list, split_name):
        split_images_dir = os.path.join(base_dir, split_name, "images")
        split_labels_dir = os.path.join(base_dir, split_name, "labels")

        for img_name in tqdm(img_list):
            # Копируем изображение
            src_img_path = os.path.join(images_dir, img_name)
            dst_img_path = os.path.join(split_images_dir, img_name)
            shutil.copy2(src_img_path, dst_img_path)

            # Пытаемся найти соответствующий txt-файл (YOLO-аннотация)
            txt_name = os.path.splitext(img_name)[0] + ".txt"
            src_txt_path = os.path.join(labels_dir, txt_name)
            if os.path.exists(src_txt_path):
                dst_txt_path = os.path.join(split_labels_dir, txt_name)
                shutil.copy2(src_txt_path, dst_txt_path)
            else:
                # Если файла с разметкой нет, можно пропустить или
                # создать пустой файл, чтобы YOLO понимал, что объектов нет
                pass

    # Копируем файлы по группам
    copy_files(train_images, "train")
    copy_files(val_images, "val")
    copy_files(test_images, "test")

    print(f"Всего изображений: {total}")
    print(f"Обучающая выборка (train): {len(train_images)}")
    print(f"Валидационная выборка (val): {len(val_images)}")
    print(f"Тестовая выборка (test): {len(test_images)}")

# Пример использования:
base_dir = "G:/researchwork/leykocytes_dataset"  # папка, в которой лежат images и labels
split_dataset(base_dir, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42)


100%|██████████| 711/711 [00:01<00:00, 558.78it/s]

Всего изображений: 7101
Обучающая выборка (train): 5680
Валидационная выборка (val): 710
Тестовая выборка (test): 711
